# Video-LLaVA 教程

本教程介绍 Video-LLaVA 视频理解模型的核心概念和使用方法。

## 目录
1. 模型概述
2. 配置与初始化
3. 视频处理
4. 时序建模
5. 完整推理流程
6. 不同模型规模对比

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import torch
from video_llava import (
    VideoLLaVAConfig,
    SamplingStrategy,
    VisionEncoder,
    TemporalTransformer,
    TemporalLSTM,
    TemporalPooling,
    VideoProjector,
    VideoLLaVA,
    VideoProcessor,
    create_video_llava,
)

print("Imports successful!")

## 1. 模型概述

Video-LLaVA 扩展 LLaVA 以处理视频时序信息：

```
视频 → 帧采样 → VisionEncoder → TemporalModel → VideoProjector → LLaMA → 输出
```

## 2. 配置与初始化

In [ ]:
# 创建配置
config = VideoLLaVAConfig(
    image_size=224,
    patch_size=16,
    vision_layers=2,
    vision_width=256,
    vision_heads=4,
    vocab_size=1000,
    hidden_size=256,
    num_layers=2,
    num_heads=4,
    intermediate_size=512,
    num_frames=4,
    temporal_mode="transformer",
    temporal_layers=1,
    temporal_heads=4,
    temporal_hidden_size=256
)

print(f"Image size: {config.image_size}")
print(f"Patch size: {config.patch_size}")
print(f"Num patches: {(config.image_size // config.patch_size) ** 2}")
print(f"Num frames: {config.num_frames}")
print(f"Temporal mode: {config.temporal_mode}")

## 3. 视频处理

### 3.1 帧采样

In [ ]:
# 模拟视频输入 [T, 3, H, W]
video = torch.randn(16, 3, 480, 640)  # 16帧, 480x640
print(f"Original video shape: {video.shape}")

# 均匀采样
sampled_uniform = VideoProcessor.sample_frames(
    video, num_frames=8, strategy=SamplingStrategy.UNIFORM
)
print(f"Uniform sampled shape: {sampled_uniform.shape}")

# 随机采样
sampled_random = VideoProcessor.sample_frames(
    video, num_frames=8, strategy=SamplingStrategy.RANDOM
)
print(f"Random sampled shape: {sampled_random.shape}")

In [ ]:
# 帧大小调整
resized = VideoProcessor.resize_frames(sampled_uniform, size=(224, 224))
print(f"Resized shape: {resized.shape}")

### 3.2 视觉编码

In [ ]:
# 创建视觉编码器
vision_encoder = VisionEncoder(config)

# 编码单帧
single_frame = torch.randn(1, 3, 224, 224)
frame_features = vision_encoder(single_frame)
print(f"Single frame features shape: {frame_features.shape}")
print(f"  - CLS token: {frame_features[:, 0, :].shape}")
print(f"  - Patch tokens: {frame_features[:, 1:, :].shape}")

## 4. 时序建模

### 4.1 Temporal Transformer

In [ ]:
# 创建时序 Transformer
temporal_transformer = TemporalTransformer(config)

# 模拟帧级特征 [B, N, D]
frame_cls_features = torch.randn(2, 4, 256)  # 2个样本, 4帧
temporal_features = temporal_transformer(frame_cls_features)
print(f"Temporal Transformer output: {temporal_features.shape}")

### 4.2 Temporal LSTM

In [ ]:
# 创建时序 LSTM
temporal_lstm = TemporalLSTM(config)

lstm_features = temporal_lstm(frame_cls_features)
print(f"Temporal LSTM output: {lstm_features.shape}")

### 4.3 Temporal Pooling

In [ ]:
# 创建时序池化
temporal_pooling = TemporalPooling(config)

# 池化需要 [B, T, P, D] 形状
patch_features = torch.randn(2, 4, 196, 256)  # 2样本, 4帧, 196 patches
pooled_features = temporal_pooling(patch_features)
print(f"Temporal Pooling output: {pooled_features.shape}")

## 5. 完整推理流程

In [ ]:
# 创建完整模型
model = VideoLLaVA(config)
print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# 准备输入
batch_size = 2
video = torch.randn(batch_size, 8, 3, 224, 224)  # 8帧视频
input_ids = torch.randint(0, 1000, (batch_size, 20))  # 20个文本token

print(f"Video shape: {video.shape}")
print(f"Input IDs shape: {input_ids.shape}")

In [ ]:
# 视频编码
with torch.no_grad():
    video_tokens = model.encode_video(video)
    print(f"Video tokens shape: {video_tokens.shape}")

In [ ]:
# 完整前向传播
with torch.no_grad():
    output = model(input_ids=input_ids, videos=video)
    
print(f"Logits shape: {output['logits'].shape}")
print(f"Video tokens shape: {output['video_tokens'].shape}")

## 6. 不同模型规模对比

In [ ]:
# 创建不同规模的模型
for size in ["tiny", "base"]:
    model = create_video_llava(model_size=size, num_frames=4)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"{size.upper():5s}: {num_params:>15,} parameters")
    print(f"       Vision layers: {model.config.vision_layers}")
    print(f"       LLM layers: {model.config.num_layers}")
    print()

In [ ]:
# 测试不同时序模式
for mode in ["transformer", "lstm", "pooling"]:
    model = create_video_llava(model_size="tiny", num_frames=4, temporal_mode=mode)
    video = torch.randn(1, 8, 3, 224, 224)
    
    with torch.no_grad():
        video_tokens = model.encode_video(video)
    
    print(f"{mode:12s}: video_tokens shape = {video_tokens.shape}")

## 总结

Video-LLaVA 的核心组件：

1. **VideoProcessor**: 帧采样和预处理
2. **VisionEncoder**: ViT 视觉编码
3. **TemporalModel**: 时序建模 (Transformer/LSTM/Pooling)
4. **VideoProjector**: 视觉到语言空间投影
5. **LLaMAModel**: 语言模型生成

关键设计选择：
- 帧采样策略：均匀采样最常用
- 时序模式：Transformer 效果最好，Pooling 最高效
- 投影类型：MLP 比 Linear 表达能力更强